In [ ]:
from bedrock_agentcore_starter_toolkit.operations.gateway.client import GatewayClient
import json
import logging
import time

## Setup Gateway

In [ ]:
def load_api_spec(file_path: str) -> list:
    with open(file_path, "r") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Expected a list in the JSON file")
    return data

In [ ]:
api_spec = load_api_spec("lambda/api_spec.json")

In [ ]:
# Initialize client
client = GatewayClient(region_name="us-west-2")
client.logger.setLevel(logging.INFO)

# Step 2.1: Create Gateway
print("Step 2.1: Creating Gateway...")
gateway = client.create_mcp_gateway(
    # the name of the Gateway - if you don't set one, one will be generated.
    name="customer-support",
    # the role arn that the Gateway will use - if you don't set one, one will be created.
    # NOTE: if you are using your own role make sure it has a trust policy that trusts bedrock-agentcore.amazonaws.com
    role_arn=None,
    # the OAuth authorization server details. If you are providing your own authorization server,
    # then pass an input of the following form: {"customJWTAuthorizer": {"allowedClients": ["<INSERT CLIENT ID>"], "discoveryUrl": "<INSERT DISCOVERY URL">}}
    authorizer_config={
        "customJWTAuthorizer": {"allowedClients": ["751st0qo85jqr2qnnjbss8p0q7"], "discoveryUrl": "https://cognito-idp.us-west-2.amazonaws.com/us-west-2_2oTa6LAsj/.well-known/openid-configuration"}
    },
    enable_semantic_search=True,
)

print(f"✓ Gateway created: {gateway['gatewayUrl']}\n")

# If role_arn was not provided, fix IAM permissions
# NOTE: This is handled internally by the toolkit when no role is provided
client.fix_iam_permissions(gateway)
print("⏳ Waiting 30s for IAM propagation...")
time.sleep(30)
print("✓ IAM permissions configured\n")

In [ ]:
# Step 2.2: Add Lambda target
print("Step 2.2: Adding Lambda target...")
lambda_target = client.create_mcp_gateway_target(
    # the gateway created in the previous step
    gateway=gateway,
    name="lambda-customer-support",
    target_type="lambda",
    target_payload={
                "lambdaArn": "arn:aws:lambda:us-west-2:407296935140:function:customer-support-stack-customer-support",
                "toolSchema": {"inlinePayload": api_spec},
    },
)
print("✓ Lambda target added\n")

In [ ]:
from bedrock_agentcore.identity.auth import requires_access_token
import asyncio

gateway_access_token = None

@requires_access_token(
    provider_name="customer-support-gateway",
    scopes=[],  # Optional unless required
    auth_flow="M2M",
)
def _get_access_token_manually(*, access_token: str):
    global gateway_access_token
    gateway_access_token = access_token
    return access_token

_get_access_token_manually(access_token="")


In [ ]:
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.tools.mcp.mcp_client import MCPClient

streamable_http_mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url="https://customer-support-3gwe5spzed.gateway.bedrock-agentcore.us-west-2.amazonaws.com/mcp", 
        # Get pat token from here: https://github.com/settings/personal-access-tokens
        headers={"Authorization": f"Bearer {gateway_access_token}"}
    )
)
# Create an agent with MCP tools
with streamable_http_mcp_client:
    # Get the tools from the MCP server
    tools = streamable_http_mcp_client.list_tools_sync()
    print(tools)

In [ ]:
# # import boto3
# # import uuid

# # client_cp = boto3.client('bedrock-agentcore-control')

# # client_dp = boto3.client('bedrock-agentcore')

# response = client_cp.create_workload_identity(
#     # name='strands-local',
# )

# response = client.get_workload_access_token(
#     workloadName='strands-local',
#     userId=uuid.uuid4().hex[:8]
# )

# response = client.get_resource_oauth2_token(
#     workloadIdentityToken=response["workloadAccessToken"],
#     resourceCredentialProviderName='customer-support-gateway',
#     scopes=[
#         'agentcore-cognito-m2m-stack-resource-server/read',
#         'agentcore-cognito-m2m-stack-resource-server/gateway'
#     ],
#     oauth2Flow='M2M',
# )